# 03 — Exploratory Data Analysis (EDA)
**Movie Review Sentiment Analysis Project**

Stage 5: EDA — Visualise dataset characteristics before modelling.

In [ ]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_ROOT)

from src.data_loader import load_imdb_dataset, get_train_test
from src.preprocessing import apply_cleaning

sns.set_theme(style='whitegrid')

df = load_imdb_dataset(PROJECT_ROOT)
train_df, test_df = get_train_test(df)
train_df['clean_review'] = apply_cleaning(train_df['review'])
train_df['word_count'] = train_df['clean_review'].str.split().str.len()
train_df['char_count'] = train_df['clean_review'].str.len()
print('Dataset loaded. Training rows:', len(train_df))

## 1. Class distribution

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
counts = train_df['sentiment'].value_counts()
sns.barplot(x=counts.index, y=counts.values, palette=['#e74c3c','#2ecc71'], ax=ax)
ax.set_title('Class Distribution in Training Set', fontsize=14)
ax.set_xlabel('Sentiment')
ax.set_ylabel('Number of Reviews')
for i, v in enumerate(counts.values):
    ax.text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()
print('Observation: The dataset is perfectly balanced — 12,500 positive and 12,500 negative.')

## 2. Review length distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Word count
for sentiment, color in [('positive','#2ecc71'), ('negative','#e74c3c')]:
    subset = train_df[train_df['sentiment'] == sentiment]['word_count']
    subset.plot.hist(bins=50, alpha=0.6, color=color, label=sentiment, ax=axes[0])
axes[0].set_title('Review Length Distribution (Word Count)')
axes[0].set_xlabel('Number of Words')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].set_xlim(0, 1500)

# Box plot
sns.boxplot(data=train_df, x='sentiment', y='word_count',
            palette={'positive':'#2ecc71','negative':'#e74c3c'}, ax=axes[1])
axes[1].set_title('Review Word Count by Sentiment')
axes[1].set_xlabel('Sentiment')
axes[1].set_ylabel('Word Count')

plt.tight_layout()
plt.show()

print(train_df.groupby('sentiment')['word_count'].describe().round(1))
print('\nObservation: Reviews range widely in length. Positive and negative'
      ' reviews have similar length distributions.')

## 3. Most frequent words by sentiment

In [ ]:
import re

# Common stopwords to exclude from word frequency (not from model)
DISPLAY_STOP = {
    'the','a','an','and','is','it','this','that','was','for','of',
    'in','to','i','s','be','are','with','as','at','by','from',
    'on','his','her','he','she','they','we','its','or','but',
    'have','has','had','do','did','one','all','my','me','him','them',
    'so','if','what','when','who','how','which','than','about','up',
    'out','into','there','their','just','were','been','would','could',
    'get','more','also','very','even','some','can','too','most','well',
    'any','each','after','before','over','here','where','while','because',
    "don't","isn't","wasn't","didn't"
}

def top_words(texts, n=20):
    words = []
    for t in texts:
        words.extend(w for w in t.split() if w not in DISPLAY_STOP and len(w) > 2)
    return Counter(words).most_common(n)

pos_words = top_words(train_df[train_df['sentiment'] == 'positive']['clean_review'])
neg_words = top_words(train_df[train_df['sentiment'] == 'negative']['clean_review'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, word_list, title, color in [
    (axes[0], pos_words, 'Top 20 Words — Positive Reviews', '#2ecc71'),
    (axes[1], neg_words, 'Top 20 Words — Negative Reviews', '#e74c3c'),
]:
    words, counts = zip(*word_list)
    sns.barplot(x=list(counts), y=list(words), color=color, ax=ax)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Count')
    ax.set_ylabel('Word')
plt.tight_layout()
plt.show()
print('Observation: Words like "great", "love", "best" appear in positive reviews;'
      '"bad", "worst", "boring" appear in negative reviews — consistent with expected sentiment.')

## 4. Word cloud (optional visualisation)

In [ ]:
try:
    from wordcloud import WordCloud
    pos_text = ' '.join(train_df[train_df['sentiment']=='positive']['clean_review'].tolist())
    neg_text = ' '.join(train_df[train_df['sentiment']=='negative']['clean_review'].tolist())
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, text, title, cmap in [
        (axes[0], pos_text, 'Positive Reviews', 'Greens'),
        (axes[1], neg_text, 'Negative Reviews', 'Reds'),
    ]:
        wc = WordCloud(width=600, height=300, background_color='white',
                       stopwords=DISPLAY_STOP, colormap=cmap,
                       max_words=100).generate(text)
        ax.imshow(wc, interpolation='bilinear')
        ax.axis('off')
        ax.set_title(title, fontsize=14)
    plt.suptitle('Word Clouds by Sentiment (Training Set)', fontsize=16)
    plt.tight_layout()
    plt.show()
except ImportError:
    print('wordcloud library not installed. Run: pip install wordcloud')